In [0]:
# If you already have added Google Service Account email on Workspace, you can skip this.

# Pull the GCP credentials securely from Databricks Secrets
gcp_client_email = dbutils.secrets.get(scope="babyname_scope", key="client_email")
gcp_project_id = dbutils.secrets.get(scope="babyname_scope", key="project_id")
gcp_private_key = dbutils.secrets.get(scope="babyname_scope", key="private_key")
gcp_private_key_id = dbutils.secrets.get(scope="babyname_scope", key="private_key_id")

# Set them as a configuration.
sc = spark.sparkContext
hconf = sc._jsc.hadoopConfiguration()
hconf.set("google.cloud.auth.service.account.enable", "true")
hconf.set("fs.gs.project.id", gcp_project_id)
hconf.set("fs.gs.auth.service.account.email", gcp_client_email)
hconf.set("fs.gs.auth.service.account.private.key", gcp_private_key)
hconf.set("fs.gs.auth.service.account.private.key.id", gcp_private_key_id)

In [0]:
# If you have skipped the above code, uncomment the following line.
# sc = spark.sparkContext

In [0]:
file_path_name = "gs://what-should-we-name-our-baby/ssa_names/*.txt" # Make sure to replace with your own file path (ensure your GCP service account has read permission)

def get_year(path):
    return int(path[-8:-4])

def parse_file(path_and_content):
    path, content = path_and_content
    year = get_year(path)
    output = []
    for line in content.splitlines():
        if line.strip():
            name, sex, count = line.split(",")
            output.append((name, sex, int(count), year))
    return output
            
name_rdd = (
    sc.wholeTextFiles(file_path_name)
    .filter(lambda x: get_year(x[0]) <= 2019)
    .flatMap(parse_file)
)

print(name_rdd.count())
print(name_rdd.take(5))

1990503
[('Mary', 'F', 7065, 1880), ('Anna', 'F', 2604, 1880), ('Emma', 'F', 2003, 1880), ('Elizabeth', 'F', 1939, 1880), ('Minnie', 'F', 1746, 1880)]


In [0]:
print(name_rdd.sortBy(lambda x: x[-1], ascending=False).take(5))

[('Olivia', 'F', 18577, 2019), ('Emma', 'F', 17226, 2019), ('Ava', 'F', 14549, 2019), ('Sophia', 'F', 13821, 2019), ('Isabella', 'F', 13407, 2019)]


In [0]:
name_rdd.saveAsTextFile("gs://what-should-we-name-our-baby/name_rdd")

In [0]:
file_path_ngram = "gs://what-should-we-name-our-baby/ngrams/*.csv"
ngram_rdd = sc.textFile(file_path_ngram).map(lambda x: x.split(","))
print(ngram_rdd.count())
print(ngram_rdd.take(5))

20500898
[['lecole', '1880', '1', '1'], ['lecole', '1881', '1', '1'], ['lecole', '1885', '1', '1'], ['lecole', '1887', '2', '2'], ['lecole', '1890', '1', '1']]


In [0]:
ngram_rdd.saveAsTextFile("gs://what-should-we-name-our-baby/ngram_rdd")